# Window + Aggregation + Watermark

In [0]:
from pyspark.sql.functions import window,col,sum as sum  

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS pyspark_catalog.moon.sales_stream_source (
    store_id     STRING,
    amount       DOUBLE,
    event_time   TIMESTAMP,
    batch_tag    STRING
) USING DELTA
""")

DataFrame[]

In [0]:
query=(spark.readStream.format('delta')\
      .table('pyspark_catalog.moon.sales_stream_source')\
     .withWatermark("event_time","10 minutes")\
    .groupBy(window("event_time","10 minutes"),"store_id")\
    .agg(sum("amount").alias("total"))\
    .writeStream\
    .format('delta')\
    .outputMode('append')\
    .trigger(availableNow=True)\
    .option('checkpointLocation','/Volumes/pyspark_catalog/moon/files/checks/')\
    .toTable('pyspark_catalog.moon.sales')
    )
query.awaitTermination()

In [0]:
spark.sql("""
INSERT INTO pyspark_catalog.moon.sales_stream_source VALUES
('S1', 100.0, '2026-08-06T10:02:00', 'batch1'),
('S1', 150.0, '2026-08-06T10:04:00', 'batch1')
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
INSERT INTO pyspark_catalog.moon.sales_stream_source VALUES
('S1', 75.0, '2026-08-06T10:25:00', 'batch2_pushes_watermark')
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

## Why no rows here
1.once window is fully finalized,means window period is closed,water mark period is also finished then only spark finalizes the window and  moves aggregated results to target space.

2.after finalizing the window if any records belongs to that window comes later(late arrive data) that will be completly ignored silently.


In [0]:
%sql
select * from pyspark_catalog.moon.sales;

window,store_id,total


In [0]:
%sql
select * from pyspark_catalog.moon.sales_stream_source 

store_id,amount,event_time,batch_tag
S1,75.0,2026-08-06T10:25:00.000Z,batch2_pushes_watermark
S1,100.0,2026-08-06T10:02:00.000Z,batch1
S1,150.0,2026-08-06T10:04:00.000Z,batch1
S1,1.0,2026-08-06T10:26:00.000Z,trigger_row


In [0]:
spark.sql("select * from pyspark_catalog.moon.sales").show(truncate=False)

+------------------------------------------+--------+-----+
|window                                    |store_id|total|
+------------------------------------------+--------+-----+
|{2026-08-06 10:00:00, 2026-08-06 10:10:00}|S1      |250.0|
+------------------------------------------+--------+-----+



In [0]:
spark.sql("""
INSERT INTO pyspark_catalog.moon.sales_stream_source VALUES
('S1', 1.0, '2026-08-06T10:26:00', 'trigger_row')
""")
 

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
INSERT INTO pyspark_catalog.moon.sales_stream_source VALUES
('S1', 1.0, '2026-08-06T10:26:00', 'batch3_trigger_flush')
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]